# Part 2: Constituency Parsing

**Ran Lu** — ran.lu@universite-paris-saclay.fr

## First part: manual analysis of sentences

1. **The cat sat on the couch when I read.**

```text
(S (NP The cat) (VP (VP sat (PP on (NP the couch))) (PP when (S (NP I) (VP read)))))
```

2. **Time flies like an arrow; fruit flies like banana.**

```text
(S (S (NP Time) (VP flies (PP like (NP an arrow)))) (S (NP fruit flies) (VP like (NP banana))))
(S (S (NP Time flies) (VP like (NP an arrow))) (S (NP fruit) (VP flies (PP like (NP banana)))))
```

3. **The spy saw the cop with the telescope.**

```text
(S (NP The spy) (VP (VP saw (NP the cop)) (PP with (NP the telescope))))
(S (NP The spy) (VP saw (NP (NP the cop) (PP with (NP the telescope)))))
```

4. **The spy saw the cop with the newspaper.**

```text
(S (NP The spy) (VP (VP saw (NP the cop)) (PP with (NP the newspaper))))
(S (NP The spy) (VP saw (NP (NP the cop) (PP with (NP the newspaper)))))
```

5. **Arabidopsis thaliana seedlings exhibit longer hypocotyls when they are grown under high ambient temperature, which is defined as thermomorphogenesis.**

```text
(S (NP Arabidopsis thaliana seedlings) (VP (VP exhibit (NP longer hypocotyls)) (PP when (S (NP they) (VP are grown (PP under (NP (NP high ambient temperature) (S (NP which) (VP is defined (PP as (NP thermomorphogenesis)))))))))))
```

6. **A spectrogram of PSN J10354824+3900279 obtained on Dec. 19.33 UT suggests that this is a type-Ia at redshift z 0.044.**

```text
(S (NP (NP A spectrogram (PP of (NP PSN J10354824+3900279))) (VP obtained (PP on (NP Dec. 19.33 UT)))) (VP suggests (PP that (S (NP this) (VP is (NP (NP a type-Ia) (PP at (NP redshift z 0.044))))))))
```

## Second part: automatic analysis

CYK omits punctuation, as in the supplied example. Phrase symbols are `S`, `NP`, `VP`, and `PP`.

In [1]:
SAMPLES = [
    {
        'text': 'The cat sat on the couch when I read.',
        'tokens': 'The cat sat on the couch when I read'.split(),
        'pos': [['DET'], ['NOUN'], ['VERB'], ['ADP'], ['DET'], ['NOUN'], ['SCONJ'], ['PRON'], ['VERB', 'NOUN']],
        'profile': 'basic',
    },
    {
        'text': 'Time flies like an arrow; fruit flies like banana.',
        'tokens': 'Time flies like an arrow fruit flies like banana'.split(),
        'pos': [['NOUN', 'VERB'], ['VERB', 'NOUN'], ['ADP', 'VERB'], ['DET'], ['NOUN'], ['NOUN'], ['VERB', 'NOUN'], ['ADP', 'VERB'], ['NOUN']],
        'profile': 'compound',
    },
    {
        'text': 'The spy saw the cop with the telescope.',
        'tokens': 'The spy saw the cop with the telescope'.split(),
        'pos': [['DET'], ['NOUN'], ['VERB', 'NOUN'], ['DET'], ['NOUN'], ['ADP'], ['DET'], ['NOUN']],
        'profile': 'pp',
    },
    {
        'text': 'The spy saw the cop with the newspaper.',
        'tokens': 'The spy saw the cop with the newspaper'.split(),
        'pos': [['DET'], ['NOUN'], ['VERB', 'NOUN'], ['DET'], ['NOUN'], ['ADP'], ['DET'], ['NOUN']],
        'profile': 'pp',
    },
    {
        'text': 'Arabidopsis thaliana seedlings exhibit longer hypocotyls when they are grown under high ambient temperature, which is defined as thermomorphogenesis.',
        'tokens': 'Arabidopsis thaliana seedlings exhibit longer hypocotyls when they are grown under high ambient temperature which is defined as thermomorphogenesis'.split(),
        'pos': [['PROPN', 'NOUN'], ['PROPN', 'ADJ', 'NOUN'], ['NOUN'], ['VERB'], ['ADJ'], ['NOUN'], ['SCONJ'], ['PRON'], ['AUX'], ['VERB', 'ADJ'], ['ADP'], ['ADJ'], ['ADJ', 'NOUN'], ['NOUN'], ['PRON'], ['AUX'], ['VERB', 'ADJ'], ['ADP'], ['NOUN']],
        'profile': 'relative',
    },
    {
        'text': 'A spectrogram of PSN J10354824+3900279 obtained on Dec. 19.33 UT suggests that this is a type-Ia at redshift z 0.044.',
        'tokens': ['A', 'spectrogram', 'of', 'PSN', 'J10354824+3900279', 'obtained', 'on', 'Dec.', '19.33', 'UT', 'suggests', 'that', 'this', 'is', 'a', 'type-Ia', 'at', 'redshift', 'z', '0.044'],
        'pos': [['DET'], ['NOUN'], ['ADP'], ['PROPN'], ['PROPN', 'NOUN'], ['VERB', 'ADJ'], ['ADP'], ['PROPN'], ['NUM'], ['PROPN'], ['VERB'], ['SCONJ'], ['PRON'], ['AUX'], ['DET'], ['NOUN', 'ADJ'], ['ADP'], ['NOUN'], ['NOUN', 'PROPN'], ['NUM']],
        'profile': 'astronomy',
    },
]

for number, sample in enumerate(SAMPLES, 1):
    ambiguous = [f"{word}={'/'.join(tags)}" for word, tags in zip(sample['tokens'], sample['pos']) if len(tags) > 1]
    print(f"{number}. {', '.join(ambiguous) if ambiguous else 'none'}")

1. read=VERB/NOUN
2. Time=NOUN/VERB, flies=VERB/NOUN, like=ADP/VERB, flies=VERB/NOUN, like=ADP/VERB
3. saw=VERB/NOUN
4. saw=VERB/NOUN
5. Arabidopsis=PROPN/NOUN, thaliana=PROPN/ADJ/NOUN, grown=VERB/ADJ, ambient=ADJ/NOUN, defined=VERB/ADJ
6. J10354824+3900279=PROPN/NOUN, obtained=VERB/ADJ, type-Ia=NOUN/ADJ, z=NOUN/PROPN


In [2]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Node:
    label: str
    left: object = None
    right: object = None
    word: str = None
    tag: str = None

    def bracket(self):
        if self.word is not None:
            leaf = f"({self.tag} {self.word})"
            return leaf if self.label == self.tag else f"({self.label} {leaf})"
        return f"({self.label} {self.left.bracket()} {self.right.bracket()})"

    def leaves(self):
        if self.word is not None:
            return [self.word]
        return self.left.leaves() + self.right.leaves()

class CYK:
    def __init__(self, grammar, max_trees=20000):
        self.lexical, self.binary = {}, {}
        self.max_trees = max_trees
        for parent, children in grammar:
            table = self.lexical if len(children) == 1 else self.binary
            table.setdefault(children, []).append(parent)

    def parse(self, tokens, pos_options):
        n = len(tokens)
        chart = [[{} for _ in range(n - width + 1)] for width in range(1, n + 1)]
        for i, (word, tags) in enumerate(zip(tokens, pos_options)):
            for tag in tags:
                for parent in self.lexical.get((tag,), []):
                    chart[0][i].setdefault(parent, []).append(Node(parent, word=word, tag=tag))
        for width in range(2, n + 1):
            for start in range(n - width + 1):
                cell = chart[width - 1][start]
                for split in range(1, width):
                    left = chart[split - 1][start]
                    right = chart[width - split - 1][start + split]
                    for left_label, left_trees in left.items():
                        for right_label, right_trees in right.items():
                            for parent in self.binary.get((left_label, right_label), []):
                                trees = cell.setdefault(parent, [])
                                room = self.max_trees - len(trees)
                                if room > 0:
                                    trees.extend(Node(parent, a, b) for a in left_trees for b in right_trees)
                                    del trees[self.max_trees:]
        return chart[-1][0].get('S', [])

CORE = [
    ('S', ('NP', 'VP')), ('NP', ('DET', 'NP')), ('NP', ('ADJ', 'NP')),
    ('VP', ('VERB', 'NP')), ('VP', ('VERB', 'PP')), ('VP', ('AUX', 'NP')),
    ('VP', ('AUX', 'VP')), ('VP', ('VP', 'NP')), ('VP', ('VP', 'PP')),
    ('PP', ('ADP', 'NP')), ('PP', ('SCONJ', 'S')),
    ('NP', ('NOUN',)), ('NP', ('PROPN',)), ('NP', ('PRON',)), ('NP', ('NUM',)),
    ('VP', ('VERB',)), ('DET', ('DET',)), ('ADJ', ('ADJ',)),
    ('ADP', ('ADP',)), ('AUX', ('AUX',)), ('SCONJ', ('SCONJ',)), ('VERB', ('VERB',)),
]

EXTRA = {
    'basic': [],
    'compound': [('S', ('S', 'S')), ('NP', ('NP', 'NP'))],
    'pp': [('NP', ('NP', 'PP'))],
    'relative': [('NP', ('NP', 'NP')), ('NP', ('NP', 'S'))],
    'astronomy': [('NP', ('NP', 'NP')), ('NP', ('NP', 'PP')), ('NP', ('NP', 'VP'))],
}

In [3]:
def first_match(trees, condition):
    return next(tree for tree in trees if condition(tree))

def selected_cyk_trees(index):
    sample = SAMPLES[index]
    options = sample['pos']
    limit = 20000
    if index == 5:
        options = [[tags[0]] for tags in options]
        limit = 150000
    trees = CYK(CORE + EXTRA[sample['profile']], limit).parse(sample['tokens'], options)
    if index == 1:
        split = [tree for tree in trees if tree.left.label == 'S' and tree.left.leaves() == sample['tokens'][:5]]
        first = first_match(split, lambda t: '(NP (NP (NOUN fruit)) (NP (NOUN flies)))' in t.right.bracket())
        second = first_match(split, lambda t: '(NP (NP (NOUN Time)) (NP (NOUN flies)))' in t.left.bracket() and t.right.bracket().startswith('(S (NP (NOUN fruit))'))
        return [first, second]
    if index in (2, 3):
        np_attach = first_match(trees, lambda t: t.right.right.label == 'NP')
        vp_attach = first_match(trees, lambda t: t.right.right.label == 'PP')
        return [vp_attach, np_attach]
    if index == 5:
        modifier = sample['tokens'][5:10]
        chosen = first_match(trees, lambda t: t.left.right is not None and t.left.right.label == 'VP' and t.left.right.leaves() == modifier)
        return [chosen]
    return [trees[0]]

for number in range(1, len(SAMPLES) + 1):
    print(f"{number}.")
    for tree in selected_cyk_trees(number - 1):
        print(tree.bracket())

1.
(S (NP (DET The) (NP (NOUN cat))) (VP (VP (VP (VERB sat)) (PP (ADP on) (NP (DET the) (NP (NOUN couch))))) (PP (SCONJ when) (S (NP (PRON I)) (VP (VERB read))))))
2.
(S (S (NP (NOUN Time)) (VP (VP (VERB flies)) (PP (ADP like) (NP (DET an) (NP (NOUN arrow)))))) (S (NP (NP (NOUN fruit)) (NP (NOUN flies))) (VP (VP (VERB like)) (NP (NOUN banana)))))
(S (S (NP (NP (NOUN Time)) (NP (NOUN flies))) (VP (VP (VERB like)) (NP (DET an) (NP (NOUN arrow))))) (S (NP (NOUN fruit)) (VP (VP (VERB flies)) (PP (ADP like) (NP (NOUN banana))))))
3.
(S (NP (DET The) (NP (NOUN spy))) (VP (VP (VP (VERB saw)) (NP (DET the) (NP (NOUN cop)))) (PP (ADP with) (NP (DET the) (NP (NOUN telescope))))))
(S (NP (DET The) (NP (NOUN spy))) (VP (VP (VERB saw)) (NP (DET the) (NP (NP (NOUN cop)) (PP (ADP with) (NP (DET the) (NP (NOUN telescope))))))))
4.
(S (NP (DET The) (NP (NOUN spy))) (VP (VP (VP (VERB saw)) (NP (DET the) (NP (NOUN cop)))) (PP (ADP with) (NP (DET the) (NP (NOUN newspaper))))))
(S (NP (DET The) (NP (NOUN s

(S (NP (NP (DET A) (NP (NP (NOUN spectrogram)) (PP (ADP of) (NP (NP (PROPN PSN)) (NP (PROPN J10354824+3900279)))))) (VP (VP (VERB obtained)) (PP (ADP on) (NP (NP (PROPN Dec.)) (NP (NP (NUM 19.33)) (NP (PROPN UT))))))) (VP (VP (VERB suggests)) (PP (SCONJ that) (S (NP (PRON this)) (VP (AUX is) (NP (DET a) (NP (NP (NOUN type-Ia)) (PP (ADP at) (NP (NP (NOUN redshift)) (NP (NP (NOUN z)) (NP (NUM 0.044))))))))))))


In [4]:
import warnings
warnings.filterwarnings('ignore')

import benepar
import spacy
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()
nlp = spacy.load('en_core_web_sm')
nlp.add_pipe('benepar', config={'model': 'benepar_en3'})

for number, sample in enumerate(SAMPLES, 1):
    print(f"{number}.")
    for sentence in nlp(sample['text']).sents:
        print(sentence._.parse_string)

1.
(S (NP (DT The) (NN cat)) (VP (VBD sat) (PP (IN on) (NP (DT the) (NN couch))) (SBAR (WHADVP (WRB when)) (S (NP (PRP I)) (VP (VBD read))))) (. .))
2.
(S (S (NP (NN Time)) (VP (VBZ flies) (PP (IN like) (NP (DT an) (NN arrow))))) (: ;) (S (NP (NN fruit)) (VP (VBZ flies) (PP (IN like) (NP (NN banana))))) (. .))
3.
(S (NP (DT The) (NN spy)) (VP (VBD saw) (NP (DT the) (NN cop)) (PP (IN with) (NP (DT the) (NN telescope)))) (. .))
4.


(S (NP (DT The) (NN spy)) (VP (VBD saw) (NP (DT the) (NN cop)) (PP (IN with) (NP (DT the) (NN newspaper)))) (. .))
5.
(S (NP (NML (NNS Arabidopsis) (NN thaliana)) (NNS seedlings)) (VP (VBP exhibit) (NP (JJR longer) (NNS hypocotyls)) (SBAR (WHADVP (WRB when)) (S (NP (PRP they)) (VP (VBP are) (VP (VBN grown) (PP (IN under) (NP (NP (JJ high) (JJ ambient) (NN temperature)) (, ,) (SBAR (WHNP (WDT which)) (S (VP (VBZ is) (VP (VBN defined) (PP (IN as) (NP (NN thermomorphogenesis)))))))))))))) (. .))
6.
(S (NP (NP (NP (DT A) (NN spectrogram)) (PP (IN of) (NP (NNP PSN) (CD J10354824) (HYPH +) (CD 3900279)))) (VP (VBN obtained) (PP (IN on) (NP (NNP Dec.) (CD 19.33) (NNP UT))))) (VP (VBZ suggests) (SBAR (IN that) (S (NP (DT this)) (VP (VBZ is) (NP (DT a) (NN type) (HYPH -) (NN Ia)) (PP (IN at) (NP (NN redshift) (NN z) (CD 0.044))))))) (. .))


## Third part: observations

| Sentences | Observation |
|---|---|
| 1 | CYK attaches the temporal clause to `VP`; benepar uses `SBAR`. |
| 2 | CYK keeps the `flies` NOUN/VERB and `like` ADP/VERB readings. Benepar selects VERB + ADP in both clauses. |
| 3–4 | Both CFGs allow PP attachment to `VP` or `NP`. Benepar chooses `VP` twice; context favors instrument (`telescope`) but possession (`newspaper`). |
| 5 | The coarse CFG recovers the nested clauses but overgenerates. Benepar attaches the relative clause inside the temperature `NP`. |
| 6 | The tailored CFG preserves the identifier and decimal. Benepar splits `+` and `-`, but correctly finds the participial modifier and `that` clause. |

`PRON` is retained because the data contain *I, they, which,* and *this*.